<a href="https://colab.research.google.com/github/ge56sob/PC-reasoning-llms-for-underrepresented-languages/blob/main/Cuong%20-%20Small%20-%20Model%203.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets huggingface_hub
!pip install -U transformers

In [ ]:


from datasets import load_dataset, get_dataset_config_names
from google.colab import userdata
from huggingface_hub import login

# Login
hf_token = userdata.get("HuggingFace").strip()
login(token=hf_token)


# Check available configs
configs = get_dataset_config_names("Qwen/PolyMath")

# Load datasets
ds_vi = load_dataset("Qwen/PolyMath", "vi")
ds_en = load_dataset("Qwen/PolyMath", "en")

#print(ds_vi)
#print(ds_en)

"""
for level in ds_vi.keys():

    print(f"\n========== {level.upper()} ==========\n")

    dataset = ds_vi[level]

    for i in range(len(dataset)):

        print(f"Row {i}")
        print("ID:", dataset[i]["id"])
        print("Question:", dataset[i]["question"])
        print("Answer:", dataset[i]["answer"])
        print("-" * 80)
"""


In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("sail/Sailor2-1B-Chat")
model = AutoModelForCausalLM.from_pretrained("sail/Sailor2-1B-Chat")


In [ ]:
import re
import torch

match_count = 0

for level in ds_vi.keys():

    print(f"\n========== {level.upper()} ==========\n")

    if level == "high":

        dataset = ds_vi[level]

        for i in range(len(dataset)):
            if i >= 0:
              question = dataset[i]["question"]
              ground_truth = str(dataset[i]["answer"]).strip()

              print("ID:", dataset[i]["id"])
              print("Question:", question)

              # Prompt
              messages = [
                  {
                      "role": "user",
                      "content":
                      question +
                      "\n\nLưu ý: Vui lòng đặt câu trảlời cuối cùng trong $\\boxed{}$."
                  }
              ]

              # Tokenize
              inputs = tokenizer.apply_chat_template(
                  messages,
                  add_generation_prompt=True,
                  tokenize=True,
                  return_dict=True,
                  return_tensors="pt",
              ).to(model.device)

              # Inference
              with torch.no_grad():

                  outputs = model.generate(
                      **inputs,
                      max_new_tokens=2000
                  )

              # Decode response
              response = tokenizer.decode(
                  outputs[0][inputs["input_ids"].shape[-1]:],
                  skip_special_tokens=True
              )

              print("Model's Full Answer:", response)

              # =========================================
              # Extract answer inside \boxed{}
              # =========================================

              match = re.search(r'\\boxed\{(.*?)\}', response)

              if match:
                  extracted_answer = match.group(1).strip()
              else:
                  extracted_answer = "NO_BOXED_ANSWER"

              # =========================================
              # Compare answers
              # =========================================

              if extracted_answer == ground_truth:
                  match_count += 1
                  print(extracted_answer + "--" + ground_truth + " (✅)")
              else:
                  print( extracted_answer + "--" + ground_truth + " (❌)")

              print("Matches: " + str(match_count))
              print("-" * 80)

# =========================================
# Final statistics
# =========================================

    total_questions = len(ds_vi["low"])

    accuracy = match_count / total_questions

    print(f"\nTotal Matches: {match_count}")
    print(f"Total Questions: {total_questions}")
    print(f"Accuracy: {accuracy:.2%}")